In [160]:
import os
from pathlib import Path
import joblib
import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px
from plotly.subplots import make_subplots

# Parametros

In [161]:
ambiente = 'dev'
costa = 'Matamoros'
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 1
RANDOM_SEED = 0
SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
DBSCAN_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
GMM_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
RF_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_energy', 'wave_period_s']
SORT_FEATURES = ["wave_energy", "wind_speed_ms", "wave_steepness"]
EXTREME_THRESHOLD = 0.9
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_clasificator/wave_clasificator_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    model_path = f'{base_path}/wave_clasificator/wave_clasificator_{{}}.pkl'
    

# Obtener datos

In [162]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime,
                {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{costa}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [163]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}

data_sample = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

In [164]:
fig = px.scatter_matrix(
    data_sample, 
    dimensions=SCALER_FEATURES
)
fig.update_layout(
    title=f'Correlación de variables de la costa {costa}',
    width=1200, 
    height=700
)
fig.show()

In [165]:
correlation_matrix = data_sample[SCALER_FEATURES].corr()
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto")
fig.update_layout(
    title=f'Matriz de correlación de variables de la costa {costa}'
)
fig.show()

# Escalar

In [166]:
scaler_path = scaler_path.format(costa)
scaler = joblib.load(scaler_path)

In [167]:
scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

# DBSCAN

In [168]:
from sklearn.cluster import DBSCAN

In [241]:
EPSILON = 0.05
MIN_SAMPLES = 10

In [242]:
dbscan = DBSCAN(
    eps=EPSILON, 
    min_samples=MIN_SAMPLES
)

In [243]:
dbscan.fit(scaled_df[DBSCAN_FEATURES])
mask_outliers = dbscan.labels_ == -1
for feature in EXTREME_FEATURES:
    mask_outliers &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

data_sample['dbscan_is_outlier'] = mask_outliers

In [244]:
data_sample['dbscan_is_outlier'].value_counts(normalize=True)*100

dbscan_is_outlier
False    97.373573
True      2.626427
Name: proportion, dtype: float64

# Grafica

In [245]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='dbscan_is_outlier',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='dbscan_is_outlier',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)
pct_outlier = (data_sample['dbscan_is_outlier'].value_counts(normalize=True)*100).loc[True]
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()

# Gaussian Mixture

In [246]:
from sklearn.mixture import GaussianMixture

In [247]:
N_CLUSTERS = 6

In [248]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [265]:
data_sample["gmm_cluster"] = gmm.fit_predict(scaled_df[GMM_FEATURES])
data_sample["gmm_cluster_probability"] = gmm.predict_proba(scaled_df[GMM_FEATURES]).max(axis=1)

In [271]:
data_sample["gmm_cluster_probability"].groupby(data_sample["gmm_cluster"]).describe()

,count,mean,std,min,25%,50%,75%,max
gmm_cluster,,,,,,,,
0,2805.0,0.964251,0.098641,0.409970,0.992893,0.999835,1.000000,1.0
1,1382.0,0.933020,0.132174,0.342961,0.953098,0.999043,0.999971,1.0
2,2100.0,0.926571,0.133888,0.363261,0.926459,0.994115,0.999431,1.0
3,839.0,0.975397,0.081284,0.453270,0.999232,1.000000,1.000000,1.0
4,2789.0,0.925155,0.132324,0.367156,0.915608,0.995234,0.999631,1.0
5,2345.0,0.933874,0.131266,0.360442,0.955330,0.998716,0.999956,1.0


In [250]:
cluster_summary = (
    data_sample.groupby("gmm_cluster")[GMM_FEATURES]
    .mean()
    .sort_values(SORT_FEATURES)
)

cluster_order = {
    old_cluster: new_cluster + 1
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

data_sample["gmm_sea_state_level"] = data_sample["gmm_cluster"].map(cluster_order)

data_sample['gmm_mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(EXTREME_THRESHOLD)
for feature in EXTREME_FEATURES[1:]:
    data_sample['gmm_mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

data_sample.loc[data_sample['gmm_mask_extremo'], 'gmm_sea_state_level'] = 7

In [251]:
data_sample['gmm_sea_state_level'].value_counts(normalize=True)*100

gmm_sea_state_level
1    22.879282
3    22.748777
5    19.127243
2    17.128874
4    11.272431
6     4.216966
7     2.626427
Name: proportion, dtype: float64

In [252]:
sea_state_names = {
    1: "Mar calmado",
    2: "Mar suave",
    3: "Mar dinámico",
    4: "Mar agitado",
    5: "Mar fuerte",
    6: "Mar peligroso",
    7: "Mar extremo"
}

data_sample["gmm_sea_state"] = data_sample["gmm_sea_state_level"].map(sea_state_names)

In [253]:
data_sample['gmm_sea_state'].value_counts(normalize=True)*100

gmm_sea_state
Mar calmado      22.879282
Mar dinámico     22.748777
Mar fuerte       19.127243
Mar suave        17.128874
Mar agitado      11.272431
Mar peligroso     4.216966
Mar extremo       2.626427
Name: proportion, dtype: float64

# Grafica

In [254]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='gmm_sea_state',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='gmm_sea_state',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)

pct_outlier = (data_sample['gmm_sea_state'].value_counts(normalize=True)*100).loc['Mar extremo']
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()

# Random Forest

In [255]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)

In [256]:
TARGET = "gmm_sea_state_level"

In [257]:
X = data_sample[RF_FEATURES]
y = data_sample[TARGET].astype(int)

In [258]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [259]:
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [260]:
y_pred = rf_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)

cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9729200652528548
Balanced accuracy: 0.9780043473426133
              precision    recall  f1-score   support

           1      0.980     0.983     0.981       701
           2      0.975     0.968     0.971       525
           3      0.966     0.971     0.969       697
           4      0.965     0.957     0.961       346
           5      0.984     0.968     0.976       586
           6      0.921     1.000     0.959       129
           7      1.000     1.000     1.000        81

    accuracy                          0.973      3065
   macro avg      0.970     0.978     0.974      3065
weighted avg      0.973     0.973     0.973      3065

[[689   4   8   0   0   0   0]
 [  7 508   7   2   1   0   0]
 [  7   6 677   2   5   0   0]
 [  0   3   4 331   3   5   0]
 [  0   0   5   8 567   6   0]
 [  0   0   0   0   0 129   0]
 [  0   0   0   0   0   0  81]]


In [261]:
feature_importance = pd.DataFrame({
    "feature": RF_FEATURES,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

,feature,importance
1,wave_energy,0.482400
2,wave_period_s,0.240902
3,wave_steepness,0.161958
0,wind_speed_ms,0.114740


In [262]:
model_path = model_path.format(costa)
joblib.dump(rf_classifier, model_path)

['c:\\Users\\guill\\Documents\\GitHub\\CienciaDeDatos\\code\\ML/wave_clasificator/wave_clasificator_Matamoros.pkl']

In [263]:
data_sample["rf_sea_state_level"] = rf_classifier.predict(X)
data_sample["rf_sea_state"] = data_sample["rf_sea_state_level"].map(sea_state_names)

# Grafica

In [264]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='rf_sea_state',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='rf_sea_state',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)

pct_outlier = (data_sample['rf_sea_state'].value_counts(normalize=True)*100).loc['Mar extremo']
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()